# Artificial Neural Network

### Importing the libraries

In [0]:
import numpy as np
import pandas as pd
import tensorflow as tf

In [2]:
tf.__version__

## Part 1 - Data Preprocessing

### Importing the dataset

In [0]:
dataset = pd.read_csv('Churn_Modelling.csv')
X = dataset.iloc[:, 3:-1].values 
#The first 3 columns are irrelevant for predicting whether a customer will leave the bank. 
#Including them would add "noise" to the model and could lead to overfitting. 
#By starting the slice at index 3, we skip these three columns
y = dataset.iloc[:, -1].values

In [4]:
print(X)

In [5]:
print(y)

### Encoding categorical data

Label Encoding the "Gender" column

In [0]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X[:, 2] = le.fit_transform(X[:, 2])

In [7]:
print(X)

One Hot Encoding the "Geography" column

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [1])], remainder='passthrough')
X = np.array(ct.fit_transform(X))
#It creates new binary columns for each category. For example:
#France becomes [1, 0, 0]
#Spain becomes [0, 1, 0]
#Germany becomes [0, 0, 1]

In [9]:
print(X)

### Splitting the dataset into the Training set and Test set

In [0]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

### Feature Scaling

In [0]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train[:, 3:] = sc.fit_transform(X_train)
X_test[:, 3:] = sc.transform(X_test)

## Part 2 - Building the ANN

### Initializing the ANN

In [0]:
ann = tf.keras.models.Sequential()
#This creates an instance of a "Sequential" model. It allows you to build the neural network layer by layer in a linear stack.

### Adding the input layer and the first hidden layer

In [0]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))
#add(): This method adds a new layer to our network.
#Dense: This class adds a fully connected layer where every neuron is connected to all neurons in the previous layer.
#units=6: This is the number of neurons in this hidden layer. It is a hyperparameter you can tune.
#activation='relu': We use the Rectified Linear Unit (ReLU) activation function for hidden layers to introduce non-linearity, allowing the model to learn complex patterns.

### Adding the second hidden layer

In [0]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))
#We add another identical hidden layer. Adding deeper layers allows the network to learn more abstract and non-linear relationships in your data.
#6 is (11+1)/2, 11 independent input features, 1 output

### Adding the output layer

In [0]:
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))
#units=1: Since this is likely a binary classification problem (e.g., Yes/No, 0/1), we only need one output neuron.
#activation='sigmoid': The sigmoid function returns a probability (a value between 0 and 1). This is perfect for predicting the likelihood of a specific outcome.

## Part 3 - Training the ANN

### Compiling the ANN

In [0]:
ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
#optimizer='adam': The optimizer updates the weights of the neurons to reduce the error. "Adam" is a very efficient version of Stochastic Gradient Descent.
#loss='binary_crossentropy': This is the mathematical formula used to calculate the error (the "loss"). Used for binary outcomes.
#metrics=['accuracy']: This tells the model to report the percentage of correct predictions during the training process.

### Training the ANN on the Training set

In [17]:
ann.fit(X_train, y_train, batch_size=32, epochs=100)
#X_train, y_train: These are your training features and your target labels.
#batch_size=32: Instead of comparing the prediction to the real result after every single row (which is slow), the model looks at 32 rows at a time before updating the weights.
#epochs=100: This means the neural network will go through the entire training dataset 100 times to refine its learning.

## Part 4 - Making the predictions and evaluating the model

### Predicting the result of a single observation

**Extra**

Use our ANN model to predict if the customer with the following informations will leave the bank: 

Geography: France

Credit Score: 600

Gender: Male

Age: 40 years old

Tenure: 3 years

Balance: \$ 60000

Number of Products: 2

Does this customer have a credit card ? Yes

Is this customer an Active Member: Yes

Estimated Salary: \$ 50000

So, should we say goodbye to that customer ?

**Solution**

In [18]:
print(ann.predict(sc.transform([[1, 0, 0, 600, 1, 40, 3, 60000, 2, 1, 1, 50000]])) > 0.5)

### Predicting the Test set results

In [19]:
y_pred = ann.predict(X_test)
y_pred = (y_pred > 0.5)

### Making the Confusion Matrix

In [20]:
from sklearn.metrics import confusion_matrix, accuracy_score

#Create confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
#[  True Positive     False Negative
#   False Positive    True Negative  ]

# Calculate Accuracy Score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy:.2%}")

write down about precision recall f1-score, why is it better than just accuracy, what are some other interesting metrics u can find

# Precision
- It is a quality metric
- Intuitively, it tells us how many predictions are which are true are actually true
- High precisions means fewer false positives
- precision = true positives/(true positives + fasle positives)

# Recall
- It is a quantity metric
- Intuitively, it tells us how many cases did the model correctly predict as true
- High recall means fewer false negatives
- recall = true postives/(true positives + false negatives)

# F1-Score
- It is the balance metric
- It is the harmonic mean of precision and recall
- It gives you a single score that punishes extreme values. If your precision is 1 but your recall is 0, your F1-Score will be 0
- F1 = 2*precision*recall/(precision + recall)

# Why are these better than just accuracy?
- For example consider out of 1000 cases, 10 are true and 990 are false and the model predicts false for all cases 
- It will have 99% accuracy. Just accuracy tells that the model is amazing
- Precision will be 0, which tells model is useless
- Recall will be 0 too, again the model is useless
- Therefore, precision, recall and F1-score helps to distinguish if the model is able to predict a rare positive or not

# Other interesting metrics
- Specificity (True Negative Rate): Measures how well the model identifies negative cases (True negative/(true negative + false positive)). Important when the cost of a false positive is very high.
- ROC-AUC Score: Measures the model's ability to distinguish between classes across all possible thresholds. A score of 1.0 is perfect; 0.5 is no better than a coin flip.
- Log Loss (Cross-Entropy Loss): Unlike accuracy, which only looks at the final label, Log Loss looks at the probability or "certainty" of the prediction. It heavily penalizes confident but wrong guesses.


